In [ ]:
import mercury as mr
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
import os

df_nodos   = pd.read_csv('data/clean/nodos_autores.csv')
df_aristas = pd.read_csv('data/clean/aristas_colaboraciones.csv')
df_autorias= pd.read_csv('data/clean/autorias_limpias.csv')

print(f'Datos cargados: {len(df_nodos):,} autores · {len(df_aristas):,} colaboraciones')

In [ ]:
anio_slider = mr.Slider(
    value=2024, min=2018, max=2024, step=1,
    label='Año de corte'
)

min_articulos = mr.Slider(
    value=3, min=1, max=20, step=1,
    label='Mínimo de artículos por autor'
)

top_inst = df_nodos['inst_principal'].value_counts().head(15).index.tolist()
inst_selector = mr.MultiSelect(
    value=top_inst[:5],
    choices=top_inst,
    label='Instituciones a destacar'
)

In [ ]:
mr.Note('''
## La ciencia no se hace sola

Cada paper científico es el resultado de una conversación — a veces entre
dos investigadores del mismo laboratorio, a veces entre instituciones en
ciudades distintas. Cuando esas conversaciones se acumulan durante años,
forman una red invisible que define qué tan conectada, diversa y potente
es la comunidad científica de un país.

Este artículo explora esa red en México entre 2018 y 2024, usando datos de
**OpenAlex**, el catálogo académico de acceso libre más grande del mundo.
La pregunta central es simple: **¿quién conecta a quién?**

Usa los controles de la izquierda para explorar distintos cortes de la red.
''')

In [ ]:
# Filtrar nodos según controles
df_n = df_nodos[df_nodos['articulos'] >= min_articulos.value].copy()

df_a = df_aristas[
    df_aristas['source'].isin(df_n['autor_id']) &
    df_aristas['target'].isin(df_n['autor_id']) &
    (df_aristas['primera_colab'] <= anio_slider.value)
].copy()

G = nx.Graph()
for _, row in df_n.iterrows():
    G.add_node(row['autor_id'],
               nombre=row['nombre'],
               articulos=int(row['articulos']),
               citas=int(row['citas_total']),
               institucion=row['inst_principal'])

for _, row in df_a.iterrows():
    G.add_edge(row['source'], row['target'], weight=int(row['peso']))

# Componente gigante
componentes = sorted(nx.connected_components(G), key=len, reverse=True)
G_giant = G.subgraph(componentes[0]).copy()

print(f'Componente gigante: {G_giant.number_of_nodes():,} nodos · {G_giant.number_of_edges():,} aristas')

In [ ]:
mr.Note('''
---
## Visualización 1: El mapa de la ciencia mexicana

Cada punto es un investigador. Cada línea es una colaboración publicada.
Los puntos **azules** son investigadores de las instituciones que seleccionaste;
el tamaño refleja cuántos artículos tiene cada persona.

Lo que verás al primer vistazo es que la red **no es un todo homogéneo**:
hay clusters bien definidos que corresponden, en su mayoría, a universidades
o centros de investigación concretos. La UNAM, el CINVESTAV y el IPN
forman galaxias propias — conectadas, pero no fusionadas.

*Pasa el cursor sobre cada punto para ver nombre e institución.*
''')

In [ ]:
pos = nx.spring_layout(G_giant, k=0.5, seed=42)

edge_x, edge_y = [], []
for u, v in G_giant.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

node_ids    = list(G_giant.nodes())
node_x      = [pos[n][0] for n in node_ids]
node_y      = [pos[n][1] for n in node_ids]
node_nombres= [G_giant.nodes[n]['nombre']      for n in node_ids]
node_inst   = [G_giant.nodes[n]['institucion'] for n in node_ids]
node_arts   = [G_giant.nodes[n]['articulos']   for n in node_ids]
node_citas  = [G_giant.nodes[n]['citas']       for n in node_ids]

colores = [
    'royalblue' if inst in inst_selector.value else '#cccccc'
    for inst in node_inst
]

fig_red = go.Figure()

fig_red.add_trace(go.Scatter(
    x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.4, color='#dddddd'),
    hoverinfo='none', showlegend=False
))

fig_red.add_trace(go.Scatter(
    x=node_x, y=node_y, mode='markers',
    marker=dict(
        size=[max(4, min(18, a**0.65)) for a in node_arts],
        color=colores, opacity=0.85,
        line=dict(width=0.5, color='white')
    ),
    text=[
        f'<b>{n}</b><br>{inst}<br>{arts} artículos · {citas:,} citas'
        for n, inst, arts, citas in zip(node_nombres, node_inst, node_arts, node_citas)
    ],
    hovertemplate='%{text}<extra></extra>',
    showlegend=False
))

fig_red.update_layout(
    title=dict(text='Red de coautoría científica en México', font=dict(size=16)),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    margin=dict(l=20, r=20, t=50, b=20),
    height=600, paper_bgcolor='white', plot_bgcolor='white'
)

fig_red.show()

In [ ]:
# Calcular métricas de centralidad
# (puede tardar ~1 min en redes grandes — normal)
print('Calculando centralidad... (puede tardar un momento)')

betweenness = nx.betweenness_centrality(G_giant, normalized=True, k=min(500, G_giant.number_of_nodes()))
degree      = dict(G_giant.degree())

df_centralidad = pd.DataFrame([
    {
        'nombre':      G_giant.nodes[n]['nombre'],
        'institucion': G_giant.nodes[n]['institucion'],
        'articulos':   G_giant.nodes[n]['articulos'],
        'citas':       G_giant.nodes[n]['citas'],
        'betweenness': betweenness[n],
        'degree':      degree[n],
    }
    for n in G_giant.nodes()
]).sort_values('betweenness', ascending=False).reset_index(drop=True)

print('Top 10 conectores:')
print(df_centralidad[['nombre','institucion','betweenness','degree']].head(10).to_string(index=False))

In [ ]:
mr.Note('''
---
## Visualización 2: Los investigadores que mueven los hilos

La **centralidad de intermediación** mide algo específico: ¿con qué frecuencia
aparece un investigador en el camino más corto entre dos colegas que, de otra
forma, no estarían conectados?

Un valor alto no significa que esa persona sea la más prolífica ni la más
citada. Significa que es un **puente**: si la retiras de la red, muchas
colaboraciones dejarían de estar conectadas. Son quienes mantienen unida,
en silencio, a la comunidad científica mexicana.

*El tamaño del punto = artículos publicados. El color = institución.*
''')

In [ ]:
df_top200 = df_centralidad.head(200).copy()

fig_conectores = px.scatter(
    df_top200,
    x='betweenness',
    y='citas',
    size='articulos',
    color='institucion',
    hover_name='nombre',
    hover_data={'degree': True, 'betweenness': ':.4f', 'institucion': True},
    labels={
        'betweenness': 'Centralidad de intermediación (betweenness)',
        'citas': 'Total de citas acumuladas',
        'articulos': 'Artículos publicados',
    },
    title='Conectores vs. Impacto — Los 200 investigadores más influyentes',
    height=520,
    template='plotly_white',
    size_max=30,
)

# Anotar top 5
for _, row in df_centralidad.head(5).iterrows():
    fig_conectores.add_annotation(
        x=row['betweenness'], y=row['citas'],
        text=row['nombre'].split()[-1],
        showarrow=True, arrowhead=2, font=dict(size=10)
    )

fig_conectores.show()

In [ ]:
mr.Note('''
---
## Visualización 3: ¿Cómo ha crecido la red con los años?

Una red científica no es estática. Cada año nuevas colaboraciones se suman —
y con ellas, nuevos puentes entre instituciones que antes no se conocían.

Las barras muestran cuántas nuevas colaboraciones surgieron cada año.
La línea roja muestra el promedio de citas de esos artículos — un proxy
del impacto que generan.

La pregunta clave: ¿publicar juntos genera más impacto? ¿O solo más volumen?
''')

In [ ]:
evolucion = (
    df_aristas
    .groupby('primera_colab')
    .agg(nuevas_colabs=('peso','count'), citas_promedio=('citas_max','mean'))
    .reset_index()
    .rename(columns={'primera_colab':'anio'})
)

inst_por_anio = (
    df_autorias
    .groupby('anio')['inst_nombre']
    .nunique()
    .reset_index()
    .rename(columns={'inst_nombre':'instituciones_activas'})
)

evolucion = evolucion.merge(inst_por_anio, on='anio', how='left')
evolucion = evolucion[evolucion['anio'] <= anio_slider.value]

fig_evol = go.Figure()

fig_evol.add_trace(go.Bar(
    x=evolucion['anio'], y=evolucion['nuevas_colabs'],
    name='Nuevas colaboraciones',
    marker_color='steelblue', opacity=0.75,
    yaxis='y1'
))

fig_evol.add_trace(go.Scatter(
    x=evolucion['anio'], y=evolucion['citas_promedio'],
    name='Citas promedio',
    mode='lines+markers',
    line=dict(color='crimson', width=2.5),
    marker=dict(size=8),
    yaxis='y2'
))

fig_evol.update_layout(
    title='Evolución de la red científica mexicana 2018–2024',
    xaxis=dict(title='Año', tickmode='linear', dtick=1),
    yaxis=dict(title='Nuevas colaboraciones', side='left'),
    yaxis2=dict(title='Citas promedio', overlaying='y', side='right', showgrid=False),
    legend=dict(x=0.05, y=0.95),
    height=450,
    template='plotly_white',
    bargap=0.3
)

fig_evol.show()

In [ ]:
mr.Note('''
---
## ¿Qué encontramos?

La red de coautoría científica mexicana es **densa en el centro, dispersa en
los bordes**: un núcleo de investigadores — principalmente de la UNAM, el
CINVESTAV y el IPN — concentra la mayor parte de las conexiones, mientras
que la periferia está poblada por autores con pocas colaboraciones y poca
visibilidad externa.

Más importante que el número de artículos es la posición en la red.
Los investigadores con mayor centralidad de intermediación son figuras que
publican con grupos de instituciones distintas — creando los puentes que
hacen que el conocimiento fluya entre comunidades que de otra forma
operarían en silos.

La pregunta que queda abierta: **¿cómo se diseñan políticas científicas
que incentiven las conexiones correctas?**

---
*Datos: OpenAlex API · Licencia CC0 · Procesado con Python / NetworkX / Plotly*
''')